# Stuff- : script in, 9:16 video out
شغّل الخلايا بالترتيب. **المدخل الوحيد المطلوب هو السكريبت (الخلية 5).**

- اختر GPU: `Runtime > Change runtime type > T4 GPU` (مهم للرندر الكامل).
- يمكن إعادة تشغيل أي خلية بأمان دون كسر الجلسة.
- تحذيرات pip عن `starlette` و`torchvision` و`google-adk` تخص حزم Colab الأخرى ولا تؤثر على المشروع.
- إن ظهر `torchvision::nms does not exist` فالخلية 2 تعالجه بحذف `torchvision` (لا يحتاجه Chatterbox).
- إن ظهرت أخطاء استيراد أخرى بعد التثبيت (مثل `numpy`): `Runtime > Restart session` ثم أعد من الخلية 1.

In [ ]:
# 1) Setup. Safe to re-run: never stays inside the folder it deletes, installs only once.
import os
os.chdir("/content")
REPO = "/content/Stuff-"
if not os.path.exists("/content/.stuff_installed"):
    !rm -rf {REPO}
    !git clone -q https://github.com/marymyc/Stuff-.git {REPO}
    !apt-get -qq install -y ffmpeg fonts-dejavu-core libsndfile1 libxi6 libxxf86vm1 libxfixes3 libgl1 libxrender1 libsm6 libegl1 > /dev/null
    print("Installing Python packages (Chatterbox + torch 2.6, takes several minutes)...")
    !pip install --progress-bar off -q -e "{REPO}[tts]" 2>&1 | tail -n 3
    open("/content/.stuff_installed", "w").write("ok")
else:
    !git -C {REPO} pull -q
# Colab ships a torchvision built for torch 2.14; Chatterbox pins torch 2.6, which makes `transformers` crash on import.
# Chatterbox does not use torchvision, so remove it (safe to repeat).
!pip uninstall -y -q torchvision
os.chdir(REPO)
print("ready in", os.getcwd())

In [ ]:
# 2) Quick environment check (GPU + Chatterbox import)
import os
os.chdir("/content/Stuff-")
!pip uninstall -y -q torchvision   # fixes "operator torchvision::nms does not exist" / "Could not import LlamaModel"
!nvidia-smi -L || echo "No GPU: use Runtime > Change runtime type"
!python -c "import torch, chatterbox; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"

In [ ]:
# 3) Blender 4.2 LTS (downloaded once)
import os
os.chdir("/content")
BL = "/content/blender-4.2.3-linux-x64"
if not os.path.exists(BL + "/blender"):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/b.tar.xz
    !tar -xf /content/b.tar.xz -C /content
os.environ["BLENDER_BIN"] = BL + "/blender"
!{BL}/blender --version | head -1
os.chdir("/content/Stuff-")

In [ ]:
# 4) (Optional) 20-second pipeline test without Blender or voice: confirms ffmpeg, captions, mixing work here
import os
os.chdir("/content/Stuff-")
!python -m stuff make /content/Stuff-/examples/mini_script.txt --voice silent --renderer pil --scale 0.25 --fps 10 --workdir /content/work/quick

In [ ]:
# 5) YOUR INPUT: the English script (about 150-170 words for a 60-70 second video)
SCRIPT = """Paste your English script here."""
# Optional: path to a clean 10+ second American-male reference clip you have rights to (locks the voice identity)
REFERENCE_WAV = ""

import os
os.chdir("/content/Stuff-")
open("/content/script.txt", "w").write(SCRIPT)
print(len(SCRIPT.split()), "words")

In [ ]:
# 6) Render the full video (Chatterbox voice + Blender scenes). Check progress in the output below.
import os
os.chdir("/content/Stuff-")
os.environ["BLENDER_BIN"] = "/content/blender-4.2.3-linux-x64/blender"
ref = f"--ref-voice {REFERENCE_WAV}" if REFERENCE_WAV else ""
!python -m stuff make /content/script.txt --renderer blender --engine CYCLES --samples 24 {ref} --workdir /content/work/final --name final

In [ ]:
# 7) Download
from google.colab import files
files.download("/content/work/final/final.mp4")